# Self-Corrective RAG (a Grading Loop)

*Optional capstone for Block 2.*

**What you will build:** the knowledge agent from 1.7, upgraded with a **self-correcting loop** — it retrieves, **grades** whether the passages are actually relevant, and if not, **rewrites the query and tries again** before answering. This combines RAG (1.7) with LangGraph cycles (2.5).

```{note}
**A word on the name.** 1.7 already made retrieval *agentic* in one sense: the agent chose *when* to search (retrieval was a tool it could call, more than once). This chapter adds the other half — the agent judges *whether what it got back was good enough* and retries if not. To keep the two ideas distinct we call this specific pattern **self-corrective RAG**; you'll also see it called "agentic RAG" or "corrective RAG (CRAG)" in the wild. Same loop, different labels.
```

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/27_agentic_rag.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)
!pip install -q sentence-transformers

## A small knowledge base (as in 1.7)

Local embeddings, no API key.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
documents = [
    "ACME return policy: items can be returned within 30 days with a receipt.",
    "ACME ships to the EU and UK; delivery takes 3 to 5 business days.",
    "ACME Pro plan costs 20 euros per month and includes priority support.",
    "Warranty: ACME hardware is covered for 2 years against manufacturing defects.",
]
doc_vectors = embedder.encode(documents, normalize_embeddings=True)

def retrieve(query, k=2):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    top = np.argsort(-(doc_vectors @ q))[:k]
    return [documents[i] for i in top]

## The graph: retrieve -> grade -> (answer | rewrite -> retrieve)

Plain RAG (1.7) retrieves once and hopes the passages are relevant. Agentic RAG adds a **grader** node and a loop: if the retrieved passages don't answer the question, it rewrites the query and retrieves again — capped so it always terminates.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    query: str
    docs: list
    relevant: bool
    tries: int
    answer: str

def do_retrieve(state: State) -> dict:
    q = state.get("query", state["question"])
    return {"docs": retrieve(q), "tries": state.get("tries", 0) + 1}

def grade(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content":
                     "Do these passages contain enough to answer the question? Reply exactly YES or NO."},
                    {"role": "user", "content": f"Question: {state['question']}\nPassages:\n" + "\n".join(state["docs"])}])
    return {"relevant": r.content.strip().upper().startswith("YES")}

def rewrite(state: State) -> dict:
    # Show the rewriter what just FAILED, so the new query steers away from it.
    # A rewriter that only sees the question can't learn from the miss — it often
    # produces a near-identical query and the loop stalls until the cap.
    failed = "\n".join(state["docs"])
    r = llm.invoke([{"role": "system", "content":
                     "Rewrite the search query to retrieve BETTER passages. You are shown the passages "
                     "the current query retrieved, which were judged NOT relevant — steer away from them "
                     "and toward the user's actual need. Output only the new query."},
                    {"role": "user", "content":
                     f"Question: {state['question']}\nIrrelevant passages just retrieved:\n{failed}"}])
    return {"query": r.content.strip()}

def answer(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Answer only from the passages. If they don't cover it, say you don't know."},
                    {"role": "user", "content": f"Question: {state['question']}\nPassages:\n" + "\n".join(state["docs"])}])
    return {"answer": r.content}

def route(state: State) -> str:
    if state["relevant"] or state["tries"] >= 2:
        return "answer"
    return "rewrite"

In [ ]:
builder = StateGraph(State)
builder.add_node("retrieve", do_retrieve)
builder.add_node("grade", grade)
builder.add_node("rewrite", rewrite)
builder.add_node("answer", answer)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", route, {"answer": "answer", "rewrite": "rewrite"})
builder.add_edge("rewrite", "retrieve")     # the loop
builder.add_edge("answer", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())

In [ ]:
out = graph.invoke({"question": "If my laptop breaks after 14 months, am I covered?"})
print("tries:", out["tries"])
print("answer:", out["answer"])

The grader turns one-shot retrieval into a small research loop — the same self-correction idea as reflection (2.5), applied to search. "agentic" RAG is just RAG with a feedback loop around it, and LangGraph makes that loop explicit and safe to run.

---

**What's next:** one more Block 2 skill — in **28** we add **long-term memory** so an agent remembers a user across separate conversations. Then **Block 3** takes an agent to production: deploy it behind a FastAPI endpoint, connect a real front end, and decide — for any task — whether the answer is raw code, PydanticAI, LangGraph, or n8n.

## Watch the loop fire

The run above answered on the first try — the warranty passage was right there, so the grader said YES and the loop never engaged. To *see* the self-correction, ask something this tiny knowledge base can't answer. Then don't run it blind — **stream it** (2.2) and watch `retrieve → grade → rewrite → retrieve → answer` happen node by node:

In [ ]:
final = {}
for chunk in graph.stream({"question": "Do you offer a student discount?"},
                          stream_mode="updates"):
    for node, update in chunk.items():
        final.update(update)
        if node == "retrieve":
            print(f"[retrieve | try {update['tries']}] {[d[:30] for d in update['docs']]}")
        elif node == "grade":
            print(f"[grade   ] relevant={update['relevant']}")
        elif node == "rewrite":
            print(f"[rewrite ] new query: {update['query']!r}")
        elif node == "answer":
            print(f"[answer  ] {update['answer'][:75]}")

print("\ntotal retrieval attempts:", final["tries"])

Read the stream: the grader says `relevant=False`, the query gets rewritten, retrieval runs again — and when the KB still has nothing (there *is* no discount doc), the cap stops the loop and the agent answers honestly instead of inventing a discount. That honest "I don't know" is the payoff: the grade node is what lets the agent notice it's under-informed rather than confabulate. On a bigger KB the same loop would often *recover* — the second, better query surfaces a passage the first one missed.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Make the loop *recover*, not just give up.** Add a document the current query misses on the first try but a rewrite could find (e.g. `"Students get 15% off any ACME plan with a valid .edu email."`) and ask "any deals for people at university?". **Done when:** you see `try 1` grade `False`, a rewrite, then `try 2` grade `True` and a correct answer — the loop earning its keep.
2. **Count the LLM calls per run.** Instrument each node with a `print` or a counter and total them for the one-shot run vs. the looping run. **Done when:** you can say how many model calls a rewrite costs (retrieve is free — it's embeddings; grade + rewrite + the final answer are the LLM calls) and finish "agentic RAG cost N× a plain lookup here."
3. **Break the grader on purpose.** Make `grade` always return `relevant=True`. **Done when:** the loop never fires, the student-discount question gets a confident wrong/empty answer, and you can state in one line what the grade node was buying you.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 —** append the doc and re-embed, then stream the vague question:

```python
documents.append("Students get 15% off any ACME plan with a valid .edu email.")
doc_vectors = embedder.encode(documents, normalize_embeddings=True)   # rebuild the index!

for chunk in graph.stream({"question": "any deals for people at university?"},
                          stream_mode="updates"):
    print(chunk)
```

The first query ("people at university") may not embed close to "students / .edu"; the rewrite ("student discount ACME plan") does. That's the recovery path — the loop turning a near-miss into a hit. (Forget the `doc_vectors` rebuild and you'll search the old index — see Common issues.)

**2 —** Per run: `retrieve` costs **0** LLM calls (pure embeddings), `grade` and `rewrite` and `answer` are **1** each. One-shot (grade passes) = 2 calls (grade + answer). One loop = 4 (grade, rewrite, grade, answer). So a rewrite roughly **doubles** the model cost of a query — the price of not confabulating.

**3 —** With `grade` hardwired to `True` the router always goes straight to `answer`: no rewrite ever runs, and the discount question gets answered from irrelevant passages. The grade node was the agent's *only* signal that its context was inadequate — remove it and you're back to plain, over-confident RAG (1.7).
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| Loop never fires in the demo | KB tiny + question well-covered → grader says YES first try | Ask something the KB can't answer (as above), or tighten the grader prompt |
| Loop always hits the cap | The KB genuinely lacks the answer, or the rewrite isn't diverging enough | The `rewrite` node here already sees the failed passages (so it *can* steer away); if it still stalls, the info isn't in the KB — the cap plus an honest "I don't know" is the correct outcome |
| New doc still not found after appending | You forgot to re-encode `doc_vectors` | Rebuild the index after changing `documents` |
| `GraphRecursionError` | Each retry is 2+ super-steps and the cap is high | Lower `tries` cap or raise `recursion_limit`, as in 2.5 |
::::